# 05 · Küme içi DEA ve girdi kontrolleri

Çıktı odaklı, iki aşamalı CRS-DEA ve VRS duyarlılığı kullanılır. Girdi aylık ortalama aktif
müşteridir: müşteri-gün / takvim günü. Çıktılar aynı ayın üç ortalama bakiyesidir.

DEA projeksiyonları büyüme tahmini veya şube bütçe hedefi değildir. Skorlar kendi kümesi içinde
görelidir. Mevcut 0,10 politikası altında uygun kümeleme modeli olmadığından hazır skorlar
**tarihsel** olarak incelenir; yeni hesap açılmaz.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


In [ ]:
import pandas as pd
DEA_CALISTIR = False
GIRDI_KONTROLLERINI_CALISTIR = False
GIRDI_DOSYASI = source_input()
DONEM = None  # None: onaylı atamalardaki son dönem
selection = qualified_selection()
decision = current_decision()
dea = latest_run("dea_verimlilik")
configs = None
display(read_json(ROOT / "python_kodlari/ayarlar/dea_v4.json"))
print("Onaylı seçim:", selection)
print("Güncel karar:", decision["status"])


## Yeni DEA hesabı

Uygun ve doğrulanmış ana model olmadan başlamaz. Eksik atama veya yetersiz küme boyutu
durumunda kısmi sonuç üretilmez. Önceki DEA klasörü üzerine yazılmaz.


In [ ]:
if DEA_CALISTIR:
    if selection is None:
        print("DEA başlatılmadı: kalite koşullarını sağlayan doğrulanmış ana model yok.")
    else:
        from verimlilik.study import run as run_dea
        configs = prepare_configs(GIRDI_DOSYASI)
        dea = run_dea(selection, configs / "dea_v4.json", DONEM, fresh_run("dea_verimlilik"))
else:
    print("Yeni DEA hesabı yapılmadı.")


In [ ]:
scores = read_table(result_path("dea_verimlilik", "scores.csv", "dea_scores.csv"))
is_current = False
if selection is not None and dea is not None:
    is_current = read_json(dea / "manifest.json").get("source_selection_run_id") == read_json(selection / "manifest.json")["run_id"]
print("Gösterilen skorlar:", "onaylı seçimle eşleşiyor" if is_current else "TARİHSEL · güncel model olarak kullanılmaz")
display(scores.head(12))
display(scores.groupby("cluster_id").efficiency.agg(["count", "mean", "min", "max"]))


In [ ]:
potentials = read_table(result_path("dea_verimlilik", "potentials.csv", "dea_potentials.csv"))
peers = read_table(result_path("dea_verimlilik", "peers.csv", "dea_peers.csv"))
SUBE = str(scores.iloc[0]["branch"])
display(potentials[potentials.branch.astype(str).eq(SUBE)])
display(peers[peers.branch.astype(str).eq(SUBE)].sort_values("lambda", ascending=False))


## Müşteri / müşteri + personel karşılaştırması

Ek girdi eklenmesi skoru mekanik olarak artırabilir; bu daha iyi tahmin veya daha doğru
verimlilik ölçümü kanıtı değildir. Kaynak kalitesi, aynı şube kapsamı, korelasyon, ±%1 yerel
veri stresi, tek emsal çıkarma ve aylık hareketler birlikte incelenir. Personel sentetiktir.
Kontrol çalışması ana DEA hesabını değiştirmez.


In [ ]:
if GIRDI_KONTROLLERINI_CALISTIR:
    if selection is None or dea is None or not is_current:
        print("Girdi kontrolü başlatılmadı: güncel onaylı seçimle eşleşen DEA gerekli.")
    else:
        from verimlilik_kontrol.audit import run as audit_inputs
        if configs is None:
            base = read_json(dea / "config.json")
            configs = prepare_configs(GIRDI_DOSYASI, {"dea_v4.json": base})
        audit = audit_inputs(configs / "dea_girdi_kontrol.json", dea, fresh_run("dea_girdi_kontrol"))
        print("Kontrol çıktısı:", audit)
else:
    print("Kontroller yeniden çalıştırılmadı; hazır tarihsel tablolar gösteriliyor.")


In [ ]:
display(read_table(result_path("dea_girdi_kontrol", "coverage.csv", "dea_audit_coverage.csv")))
display(read_table(result_path("dea_girdi_kontrol", "comparisons.csv", "dea_audit_comparisons.csv")))
display(read_table(result_path("dea_girdi_kontrol", "stress_summary.csv", "dea_audit_stress.csv")))
